In [1]:
pip install datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.5 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 40.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

In [ ]:
from datasets import load_dataset


dataset2 = load_dataset("Estwld/empathetic_dialogues_llm")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/806k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/798k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19533 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/2770 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [ ]:

train_data2 = dataset2["train"]

In [ ]:
def format_conversation2(example):
    texts = []
    context = ""

    conv = example["conversations"]

    for i in range(len(conv)):
        role = "<|User|>" if conv[i]["role"] == "user" else "<|Assistant|>"
        context += f"{role} {conv[i]['content']}\n"

        # Only create sample when assistant speaks
        if conv[i]["role"] == "assistant":
            texts.append(context.strip())

    return {"text": texts}

In [ ]:
formatted_training_data2 = []

for i in range(len(train_data2)):
    result = format_conversation2(train_data2[i])
    formatted_training_data2.extend(result["text"])  # ✅ FIX

In [ ]:
formatted_training_data2[3]

"<|User|>  it feels like hitting to blank wall when i see the darkness\n<|Assistant|> Oh ya? I don't really see how"

In [ ]:
dataset1 = load_dataset("nbertagnolli/counsel-chat")

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


20220401_counsel_chat.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

In [ ]:
train_data1 = dataset1["train"]

In [ ]:
print(train_data1[0])

{'questionID': 0, 'questionTitle': 'Do I have too many issues for counseling?', 'questionText': 'I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?', 'questionLink': 'https://counselchat.com/questions/do-i-have-too-many-issues-for-counseling', 'topic': 'depression', 'therapistInfo': 'Jennifer MolinariHypnotherapist & Licensed Counselor', 'therapistURL': 'https://counselchat.com/therapists/jennifer-molinari', 'answerText': 'It is very common for\xa0people to have multiple issues that they want to (and need to) address in counseling.\xa0 I have had clients ask that same question and through more exploration, there is often an underlying fear that they\xa0 "can\'t be h

In [ ]:
def clean_text(text):
    if text is None:
        return ""
    return text.replace("\xa0", " ").replace("\n", " ").strip()
def format_counsel(example):
    question = clean_text(example["questionText"])
    answer = clean_text(example["answerText"])

    return f"<|User|> {question}\n<|Assistant|> {answer}"

formatted_training_data1 = []

for i in range(len(train_data1)):
    text = format_counsel(train_data1[i])
    formatted_training_data1.append(text)

In [ ]:
formatted_training_data1[0]

'<|User|> I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.    I’ve never had counseling about any of this. Do I have too many issues to address in counseling?\n<|Assistant|> It is very common for people to have multiple issues that they want to (and need to) address in counseling.  I have had clients ask that same question and through more exploration, there is often an underlying fear that they  "can\'t be helped" or that they will "be too much for their therapist." I don\'t know if any of this rings true for you. But, most people have more than one problem in their lives and more often than not,  people have numerous significant stressors in their lives.  Let\'s face it, life can be complicated! Therapists are completely ready and equipped to handle all of the 

In [ ]:
combined_texts = formatted_training_data2 + formatted_training_data1

In [ ]:
from datasets import Dataset

combined_dataset = Dataset.from_dict({
    "text": combined_texts
})

In [ ]:
print(combined_dataset[0])

{'text': '<|User|> I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.\n<|Assistant|> Was this a friend you were in love with, or just a best friend?'}


In [ ]:
combined_dataset = combined_dataset.shuffle(seed=42)

In [ ]:
length =  len(combined_dataset)
length

43028

In [ ]:
#combined_dataset = combined_dataset.select()

In [ ]:

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
lengths = []

for i in range(1000):  # check first 1000 samples
    tokens = tokenizer(combined_dataset[i]["text"])
    lengths.append(len(tokens["input_ids"]))

print("Max length:", max(lengths))
print("Average length:", sum(lengths) / len(lengths))

Max length: 547
Average length: 81.535


In [ ]:
print(len(tokenizer))

50257


In [ ]:
tokenizer.add_special_tokens({
    "additional_special_tokens": ["<|User|>", "<|Assistant|>"]
})

2

In [ ]:
print(len(tokenizer))

50259


In [ ]:
model.resize_token_embeddings(len(tokenizer))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50259, 768)

In [ ]:
print(model.get_input_embeddings().weight.shape)

torch.Size([50259, 768])


In [ ]:
print(tokenizer.special_tokens_map)

{'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}


In [ ]:
def tokenize_and_mask(example):
    tokenizer.pad_token = tokenizer.eos_token
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    input_ids = tokenized["input_ids"]
    labels = input_ids.copy()

    # Decode to tokens (string form)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    learning = False  # whether we are inside Assistant response

    for i, token in enumerate(tokens):
        # GPT2 splits words → so match partial tokens
        if "<|Assistant|>" == token:
            learning = True
            labels[i] = -100
            continue
        elif "<|User|>" == token:
            learning = False

        if not learning:
            labels[i] = -100

    tokenized["labels"] = labels
    return tokenized

In [ ]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
50256


In [ ]:
tokenized_dataset = combined_dataset.map(tokenize_and_mask)

Map:   0%|          | 0/43028 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset[0])

{'text': "<|User|> I'm so not prepared for this presentation at work tomorrow\n<|Assistant|> Oh no, I have been there! I hope everything goes okay!\n<|User|> Thank you. I do too but I'm so worried about messing up.\n<|Assistant|> Don't worry about it, worrying will only make things worse.", 'input_ids': [50257, 314, 1101, 523, 407, 5597, 329, 428, 10470, 379, 670, 9439, 198, 50258, 3966, 645, 11, 314, 423, 587, 612, 0, 314, 2911, 2279, 2925, 8788, 0, 198, 50257, 6952, 345, 13, 314, 466, 1165, 475, 314, 1101, 523, 7960, 546, 37241, 510, 13, 198, 50258, 2094, 470, 5490, 546, 340, 11, 18916, 481, 691, 787, 1243, 4785, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50

In [ ]:
print(tokenizer.special_tokens_map)
print(tokenizer.pad_token)
print(model.get_input_embeddings().weight.shape)

{'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}
<|endoftext|>
torch.Size([50259, 768])


In [ ]:
print(tokenizer.convert_tokens_to_ids("<|User|>"))
print(tokenizer.convert_tokens_to_ids("<|Assistant|>"))

50257
50258


In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
!pip uninstall -y torchao
!pip install torchao>=0.16.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install torchao>=0.16.0

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT-2 attention layer
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 294,912 || all params: 124,736,256 || trainable%: 0.2364


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,)

In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,6.408785
20,4.963378
30,3.075682
40,1.485204
50,1.141442
60,1.114050
70,1.004771
80,0.895899
90,1.004371
100,0.991689


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("chatbot-lora-final")
tokenizer.save_pretrained("chatbot-lora-final")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('chatbot-lora-final/tokenizer_config.json',
 'chatbot-lora-final/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_name = "gpt2"

# Load tokenizer FIRST
tokenizer = AutoTokenizer.from_pretrained("chatbot-lora-final")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

#Resize
base_model.resize_token_embeddings(len(tokenizer))

# Now load LoRA
model = PeftModel.from_pretrained(base_model, "chatbot-lora-final")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50259, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
           

In [ ]:
input_text = "<|User|> I feel really depressed\n<|Assistant|>That is so sad. It's hard to describe it, but you are definitely not alone!<|User|>I got kicked out from job \n <|Assistant|>"

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id
)

# 🔥 Only generated part
generated_ids = output[0][inputs["input_ids"].shape[-1]:]

print(tokenizer.decode(generated_ids, skip_special_tokens=True))

 That's terrible! What did they do?
